# Fine-tuning Whisper with LoRA — the raw PyTorch loop

This notebook walks the training loop behind `python -m src.asr.train` in
[this repo](https://github.com/Rababb-P/CustomVoiceAgent): **whisper-small**,
LoRA adapters on the attention projections, mixed precision, gradient
accumulation, and best-epoch selection — no `Trainer`, every step visible.

The training data is fully synthetic: LLM-written sentences containing my
domain vocabulary (*WATonomous, Reparo, YOLOv11, ...*), rendered by ten Kokoro
TTS voices with speed/noise augmentation, mixed with real Common Voice speech
so the model doesn't forget general English. Validation is honest: held-out
sentences spoken by held-out voices, so it measures generalization, not
memorization (see `src/asr/gen_sentences.py` / `synthesize.py` / `prepare_data.py`).

> Demo run: to keep this notebook executable in minutes, it trains **1 epoch on
> a 200-example subset**. The committed production model used the identical
> loop via `python -m src.asr.train` (3 epochs, 2 784 examples).

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import torch
from src.config import load_config

cfg = load_config(str(REPO / "configs/asr_finetune.yaml"))
device = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = device == "cuda"
torch.manual_seed(cfg["training"].get("seed", 42))
print(f"device={device}  torch={torch.__version__}")

device=cuda  torch=2.11.0+cu128


## 1. Data

`prepare_data.py` already assembled the dataset: 16 kHz mono audio as plain
`{array, sampling_rate}` columns plus a transcript. Whisper never sees raw
audio — it eats 80-bin log-mel spectrograms, which we compute below.

In [2]:
from datasets import load_from_disk

SUBSET = 200  # demo-sized; the real run uses all of it

ds = load_from_disk(str(REPO / cfg["data"]["processed_dir"] / "hf_dataset"))
print(ds)
ds["train"] = ds["train"].select(range(SUBSET))

ex = ds["train"][0]
print(f"\ntranscript: {ex['transcript']!r}")
print(f"audio: {len(ex['audio']['array'])} samples @ {ex['audio']['sampling_rate']} Hz "
      f"({len(ex['audio']['array'])/ex['audio']['sampling_rate']:.1f}s)")

C:\Users\rabab\Desktop\customvoiceagent\CustomVoiceAgent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['audio', 'transcript'],
        num_rows: 2784
    })
    validation: Dataset({
        features: ['audio', 'transcript'],
        num_rows: 51
    })
})

transcript: 'His father was Jewish and his mother was Irish Catholic.'
audio: 75264 samples @ 16000 Hz (4.7s)


## 2. Model + LoRA

Full fine-tuning whisper-small means updating 245 M parameters — too much for
8 GB of VRAM and unnecessary. LoRA freezes the base model and injects low-rank
update matrices into the attention query/value projections: **1.4 % of the
parameters train**, and afterwards the adapters merge back into the base
weights so inference costs nothing extra.

In [3]:
from peft import LoraConfig, get_peft_model
from transformers import WhisperForConditionalGeneration, WhisperProcessor

base = cfg["model"]["base"]
processor = WhisperProcessor.from_pretrained(
    base, language=cfg["model"]["language"], task=cfg["model"]["task"]
)
model = WhisperForConditionalGeneration.from_pretrained(base)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

lora = cfg["lora"]
model = get_peft_model(model, LoraConfig(
    r=lora["r"], lora_alpha=lora["alpha"], lora_dropout=lora["dropout"],
    target_modules=lora["target_modules"],
))
model.print_trainable_parameters()
model = model.to(device)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:  91%|█████████▏| 438/479 [00:00<00:00, 4364.81it/s]

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 4244.45it/s]

trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429


## 3. Features + batching

Each example becomes `input_features` (the log-mel spectrogram) and `labels`
(tokenized transcript). The collator pads features and labels separately and
masks label padding with `-100` so the loss ignores it — reused straight from
`src/asr/train.py`.

In [4]:
from torch.utils.data import DataLoader

from src.asr.train import Collator


def preprocess(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["transcript"]).input_ids
    return batch


ds = ds.map(preprocess, remove_columns=ds["train"].column_names)  # in-process:
# datasets' worker pool can't see notebook globals like `processor`
ds.set_format("torch", columns=["input_features", "labels"], output_all_columns=True)

t = cfg["training"]
collate = Collator(processor)
train_loader = DataLoader(ds["train"], batch_size=t["per_device_train_batch_size"],
                          shuffle=True, collate_fn=collate)
val_loader = DataLoader(ds["validation"], batch_size=t["per_device_train_batch_size"],
                        collate_fn=collate)
len(train_loader), len(val_loader)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   1%|          | 2/200 [00:00<00:17, 11.16 examples/s]

Map:   2%|▏         | 4/200 [00:00<00:15, 12.60 examples/s]

Map:   3%|▎         | 6/200 [00:00<00:15, 12.61 examples/s]

Map:   4%|▍         | 8/200 [00:00<00:15, 12.77 examples/s]

...

Map:  92%|█████████▏| 47/51 [00:03<00:00, 11.90 examples/s]

Map:  96%|█████████▌| 49/51 [00:03<00:00, 11.30 examples/s]

Map: 100%|██████████| 51/51 [00:04<00:00, 10.48 examples/s]

Map: 100%|██████████| 51/51 [00:04<00:00, 11.92 examples/s]

(25, 7)

## 4. Optimizer, schedule, mixed precision

- **AdamW** over the trainable (LoRA) parameters only.
- **Linear warmup → linear decay**, written as a plain `LambdaLR`. The peak LR
  matters more than anything here: an earlier run at `1e-3` fine-tuned to
  2.8 % WER after epoch 1, then *destabilized to 28.6 %* mid-run. `2e-4`
  trains the same adapters smoothly.
- **fp16 autocast + GradScaler** halves memory and roughly doubles throughput
  on consumer GPUs; the scaler guards against fp16 gradient underflow.

In [5]:
EPOCHS = 1  # demo; config says 3 for the real run
accum = t["gradient_accumulation_steps"]
steps_per_epoch = -(-len(train_loader) // accum)  # ceil
total_steps = steps_per_epoch * EPOCHS


def lr_lambda(step):  # linear warmup, then linear decay to zero
    if step < t["warmup_steps"]:
        return step / max(1, t["warmup_steps"])
    return max(0.0, (total_steps - step) / max(1, total_steps - t["warmup_steps"]))


optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad), lr=float(t["learning_rate"])
)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler(device, enabled=use_fp16)
total_steps

13

## 5. The loop

Forward → scaled backward → (every `accum` batches) clip, step, update. That's
all `Trainer.train()` does — owning it means each piece is inspectable, and the
gradient-accumulation trick (effective batch 16 on an 8 GB card) is explicit.

In [6]:
from time import perf_counter

model.train()
step, t0 = 0, perf_counter()
for epoch in range(1, EPOCHS + 1):
    optimizer.zero_grad(set_to_none=True)
    for i, batch in enumerate(train_loader):
        features = batch["input_features"].to(device)
        labels = batch["labels"].to(device)
        with torch.autocast(device, enabled=use_fp16):
            loss = model(input_features=features, labels=labels).loss / accum
        scaler.scale(loss).backward()

        if (i + 1) % accum == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            step += 1
            if step % 5 == 0:
                print(f"step {step:3d}/{total_steps}  loss {loss.item()*accum:.4f}  "
                      f"lr {scheduler.get_last_lr()[0]:.2e}  "
                      f"({(perf_counter()-t0)/step:.1f}s/step)")

step   5/13  loss 3.1424  lr 2.00e-05  (1.6s/step)


step  10/13  loss 3.5467  lr 4.00e-05  (1.4s/step)


## 6. Evaluate: word error rate on held-out voices

Greedy decode the validation split — sentences *and* voices the model never
saw — and score with `jiwer`. (In `train.py` this runs per epoch and only the
best-WER epoch's adapter is saved: a bad final epoch can't overwrite a good
earlier one.)

In [7]:
import jiwer

model.eval()
preds, refs = [], []
with torch.no_grad():
    for batch in val_loader:
        features = batch["input_features"].to(device, dtype=model.dtype)
        out = model.generate(input_features=features,
                             max_length=t["generation_max_length"])
        preds.extend(processor.batch_decode(out, skip_special_tokens=True))
        labels = batch["labels"].masked_fill(
            batch["labels"] == -100, processor.tokenizer.pad_token_id)
        refs.extend(processor.batch_decode(labels, skip_special_tokens=True))

wer = jiwer.wer(refs, preds)
print(f"val WER after {EPOCHS} epoch on {SUBSET} examples: {wer:.2%}")
for r, p in list(zip(refs, preds))[:3]:
    print(f"\n  ref:  {r}\n  pred: {p}")

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


val WER after 1 epoch on 200 examples: 9.05%

  ref:  The WATonomous perception team meets twice a week to review detection regressions.
  pred:  The WA-Tonimus Perception Team meets twice a week to review detection regressions.

  ref:  WATonomous competition season means the whole team basically lives in the bay.
  pred:  W.A. Tarnamus competition, season means the whole team basically lives in the bay.

  ref:  The trickiest part of WATonomous perception is objects the training set never saw.
  pred:  The trickiest part of W.A. Tarnamus' perception is objects the training set never saw.


## 7. Results of the full run

The identical loop, run to completion (`python -m src.asr.train`, 3 epochs,
2 784 examples), merged and exported to CTranslate2 (`make export-asr`),
evaluated on 51 held-out-voice clips against two baselines (`make eval-asr`):

| model | WER | CER | custom-vocab terms @100% |
|---|---|---|---|
| whisper-small (base) | 8.92% | 3.06% | 3/8 (*WATonomous: 0%*) |
| base + hotword biasing | 7.72% | 2.60% | 4/8 |
| **LoRA fine-tune (this loop)** | **2.40%** | **0.81%** | **8/8** |

The fine-tune has to beat not just base Whisper but base **plus** hotword
biasing (the cheap trick) to justify existing — and it does, by 3×.